# 聚合 hook：wrap_tool_call 的多hook子组合

`wrap_tool_call` 参数只接受**一个**钩子函数（`ToolCallWrapper | None`）。
要叠加多个关注点（审计 + 耗时 + 统计），有两种做法：

| 方案 | 写法 | 适用 |
|---|---|---|
| 手动聚合 | `compose(*hooks)` 自制组合器，洋葱顺序（先列的在外层） | 自建 `StateGraph + ToolNode`（LangGraph 本体无官方多钩子） |
| 官方实现 | `create_agent(middleware=[M1(), M2()])`，框架自动组合（先定义 = 外层） | 接受 langchain agents 全家桶时白拿组合 + LangSmith 追踪 |

本节复用 `9_wrap_tool_call-观测埋点.ipynb` 的三个钩子：`timed`（耗时）、`audit`（审计）、`count_result`（成败统计）。

In [1]:
import time
from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt.tool_node import ToolCallRequest  # 注意：不在 prebuilt 顶层导出
from rich import print

load_dotenv(override=True)

model = init_chat_model(
    model_provider="deepseek",
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)


# 定义工具（sleep 模拟耗时，让耗时统计有内容可看）
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """根据城市名称查询天气

    Args:
        city (str): 城市名称

    Returns:
        str: 城市天气情况
    """
    time.sleep(2)
    return f"{city} 的天气是晴天，温度 25°C"


@tool(parse_docstring=True)
def get_news(topic: Literal["科技", "体育", "娱乐"]) -> str:
    """根据主题查询新闻

    Args:
        topic (Literal["科技", "体育", "娱乐"]): 新闻主题

    Returns:
        str: 新闻内容
    """
    return {
        "科技": "最新科技新闻：AI 技术正在快速发展。",
        "体育": "最新体育新闻：中国队取得了胜利。",
        "娱乐": "最新娱乐新闻：歌手发布了新专辑。",
    }[topic]


tools = [get_weather, get_news]
model_with_tools = model.bind_tools(tools)


# ===== 9 号笔记的三个钩子，原样复用 =====
def timed(request: ToolCallRequest, execute) -> ToolMessage | object:
    """耗时统计"""
    t0 = time.perf_counter()
    result = execute(request)
    cost = (time.perf_counter() - t0) * 1000
    print(f"[耗时] {request.tool_call['name']}: {cost:.0f}ms")
    return result


def audit(request: ToolCallRequest, execute) -> ToolMessage | object:
    """工具审计"""
    tc = request.tool_call
    thread_id = request.runtime.config["configurable"]["thread_id"]
    print(f"[审计] thread_id: {thread_id}, {tc['name']} : {tc['args']}")
    res = execute(request)
    print(f"[审计] thread_id: {thread_id}, {tc['name']} : {res.status}")
    return res


result_dict: dict[str, dict[str, int]] = {}


def count_result(request: ToolCallRequest, execute) -> ToolMessage | object:
    """成败统计：{工具名: {status: 次数}}"""
    tc = request.tool_call
    res = execute(request)
    counts = result_dict.setdefault(tc["name"], {})
    counts[res.status] = counts.get(res.status, 0) + 1
    return res


# ===== 手动聚合：compose 组合器 =====
def compose(*hooks):
    """把多个钩子组合成一个。先列的在外层：最先执行、最后收尾（洋葱顺序）"""

    def wrapped(request, execute):
        for hook in reversed(hooks):  # 从内往外包：最后列的最贴近工具
            execute = lambda req, h=hook, nxt=execute: h(req, nxt)
        return execute(request)

    return wrapped


tool_node = ToolNode(tools, wrap_tool_call=compose(audit, timed, count_result))


class ChatState(MessagesState):
    pass


def llm_node(state: ChatState) -> dict:
    return {"messages": [model_with_tools.invoke(state["messages"])]}


def router(state: ChatState) -> Literal["tool_node", END]:
    return "tool_node" if state["messages"][-1].tool_calls else END


builder = StateGraph(state_schema=ChatState)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", router, [END, "tool_node"])
builder.add_edge("tool_node", "llm_node")
graph = builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "1"}}
res = graph.invoke(
    {"messages": [HumanMessage("帮我查一下北京的天气")]}, config=config)

print("\n--- 聚合效果 ---")
print("成败统计:", result_dict)

[审计] thread_id: 1, get_weather : {'city': '北京'}

[耗时] get_weather: 2001ms

[审计] thread_id: 1, get_weather : success

--- 聚合效果 ---

成败统计:
{'get_weather': {'success': 1}}

# 2. 官方实现：create_agent + middleware 列表

每个关注点一个 `AgentMiddleware` 子类，各自实现 `wrap_tool_call`，
以列表传入 `create_agent`——框架收集后自动链式组合（官方 `_chain_tool_call_wrappers`，先定义 = 外层），
且每层自动包 traceable span（LangSmith 可观测）。

In [2]:
# 官方实现：AgentMiddleware 子类 + middleware 列表，框架自动组合（先定义 = 外层）
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware


class AuditMiddleware(AgentMiddleware):
    """审计：放在列表第一位 = 外层"""

    def wrap_tool_call(self, request: ToolCallRequest, handler):
        tc = request.tool_call
        print(f"[审计] {tc['name']} : {tc['args']}")
        result = handler(request)
        print(f"[审计] {tc['name']} : {result.status}")
        return result


class TimedMiddleware(AgentMiddleware):
    """耗时"""

    def wrap_tool_call(self, request: ToolCallRequest, handler):
        t0 = time.perf_counter()
        result = handler(request)
        cost = (time.perf_counter() - t0) * 1000
        print(f"[耗时] {request.tool_call['name']}: {cost:.0f}ms")
        return result


class CountMiddleware(AgentMiddleware):
    """成败统计：状态放在实例上，不污染全局"""

    def __init__(self):
        super().__init__()
        self.stats: dict[str, dict[str, int]] = {}

    def wrap_tool_call(self, request: ToolCallRequest, handler):
        tc = request.tool_call
        result = handler(request)
        counts = self.stats.setdefault(tc["name"], {})
        counts[result.status] = counts.get(result.status, 0) + 1
        return result


count_middleware = CountMiddleware()
agent = create_agent(
    model, tools,
    middleware=[AuditMiddleware(), TimedMiddleware(), count_middleware],
    checkpointer=InMemorySaver(),
)

res = agent.invoke(
    {"messages": [HumanMessage("帮我查一下北京的天气")]},
    config={"configurable": {"thread_id": "2"}})

print("\n--- 聚合效果 ---")
print("成败统计:", count_middleware.stats)

[审计] get_weather : {'city': '北京'}

[耗时] get_weather: 2001ms

[审计] get_weather : success

--- 聚合效果 ---

成败统计:
{'get_weather': {'success': 1}}

## 实测踩到的坑

1. **钩子里不能吞异常**：多个钩子串联时，任何一层 `except Exception` 不 `raise`，
   都会吞掉 `GraphInterrupt`（审批失效）或改变 `handle_tool_errors` 的行为——
   官方 deepagents 项目同样用注释提醒"middleware 必须放行 GraphBubbleUp"。
2. **两个方案不能混搭**：`compose` 用于自建图；`middleware` 列表只对 `create_agent` 生效，
   后者内部就是官方版的 compose（`_chain_tool_call_wrappers`，first = outermost），
   外加每层 traceable span（LangSmith 追踪）。
3. 顺序有讲究：审计/权限放外层（能记录完整耗时、拒绝时不启动计时），
   与工具强相关的改写放内层。